# Step 5: MesoNet Deepfake Detection Training
## Train MesoNet model on 15,570 FaceForensics images

This notebook trains a MesoNet model specifically designed for deepfake detection.

## Setup: Mount Google Drive and Download Data

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive')
print("Google Drive mounted successfully")

## Extract Data from Google Drive

In [ ]:
import zipfile
import os

print("Extracting CelebDF.zip from Google Drive...")
with zipfile.ZipFile('/content/drive/MyDrive/CelebDF.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("✅ Extraction complete!")

## Verify Data Extraction

In [ ]:
import os

print("=" * 60)
print("DATA EXTRACTION DIAGNOSTIC")
print("=" * 60)

print("\n1️⃣ Contents of /content/:")
try:
    contents = os.listdir('/content/')
    for item in sorted(contents):
        full_path = os.path.join('/content/', item)
        if os.path.isdir(full_path):
            print(f"   📁 {item}/")
        else:
            print(f"   📄 {item}")
except Exception as e:
    print(f"   ❌ ERROR: {e}")

print("\n2️⃣ Checking for CelebDF directories:")
paths_to_check = [
    '/content/CelebDF',
    '/content/drive/MyDrive/CelebDF',
    '/content/CelebDF/CelebDF'
]

for path in paths_to_check:
    if os.path.exists(path):
        print(f"   ✅ Found: {path}/")
        # Show subdirectories
        try:
            for item in os.listdir(path):
                sub_path = os.path.join(path, item)
                if os.path.isdir(sub_path):
                    print(f"      📁 {item}/")
        except:
            pass
    else:
        print(f"   ❌ NOT found: {path}/")

print("\n3️⃣ ZIP file verification:")
if os.path.exists('/content/drive/MyDrive/CelebDF.zip'):
    size_gb = os.path.getsize('/content/drive/MyDrive/CelebDF.zip') / (1024**3)
    print(f"   ✅ Found: /content/drive/MyDrive/CelebDF.zip ({size_gb:.2f} GB)")
else:
    print(f"   ❌ NOT found: CelebDF.zip")

In [ ]:
import subprocess
import shutil

# Check for backslash issues and fix them
print("Checking for backslash-named folders...")
result = subprocess.run(['ls', '-la', '/content/'], capture_output=True, text=True)
print(result.stdout)

# If CelebDF with backslashes exists, rename it
try:
    # Try to find and rename folder with backslashes
    subprocess.run(['find', '/content/', '-maxdepth', '1', '-name', '*CelebDF*', '-type', 'd'], 
                   capture_output=True, text=True)
    
    # Better: just verify the structure
    import glob
    celebdf_folders = glob.glob('/content/CelebDF*')
    print(f"\nFound CelebDF folders: {celebdf_folders}")
    
    if celebdf_folders:
        # Use the first one found
        src = celebdf_folders[0]
        if src != '/content/CelebDF':
            print(f"Renaming {src} to /content/CelebDF...")
            shutil.move(src, '/content/CelebDF')

except Exception as e:
    print(f"Error: {e}")

# Final check
import os
if os.path.exists('/content/CelebDF'):
    print("\n✅ /content/CelebDF exists!")
    print("Contents:", os.listdir('/content/CelebDF'))
    if os.path.exists('/content/CelebDF/train'):
        print(f"  Train images: {len(os.listdir('/content/CelebDF/train'))}")
    if os.path.exists('/content/CelebDF/validation'):
        print(f"  Validation folders: {os.listdir('/content/CelebDF/validation')}")
else:
    print("\n❌ /content/CelebDF still not found")

## Install Requirements

In [ ]:
!pip install tensorflow opencv-python pillow numpy matplotlib scikit-learn tqdm -q
print("Dependencies installed")

## Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from sklearn.metrics import classification_report, confusion_matrix
import cv2

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.test.is_built_with_cuda()}")

## Define MesoNet Architecture

In [ ]:
def create_mesonet():
    """
    MesoNet: A Compact Facial Video Forgery Detection Network
    Designed specifically for deepfake detection
    """
    model = keras.Sequential([
        # Block 1
        layers.Conv2D(16, (3, 3), padding='same', activation='relu', 
                     input_shape=(256, 256, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(16, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.1),
        
        # Block 2
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.1),
        
        # Block 3
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        # Fully Connected Layers
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        # Output
        layers.Dense(1, activation='sigmoid')  # Binary classification: Real=0, Fake=1
    ])
    
    return model

model = create_mesonet()
model.summary()

## Compile Model

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', 
             keras.metrics.Precision(),
             keras.metrics.Recall(),
             keras.metrics.AUC()]
)
print("Model compiled successfully")

## Prepare Data Generators

In [ ]:
# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

# No augmentation for validation
val_datagen = ImageDataGenerator(rescale=1./255)

# Path to your data
TRAIN_DIR = '/path/to/train/data'  # Change this to your path
VAL_DIR = '/path/to/validation/data'  # Change this to your path

print("Data generators created")

## Load and Prepare Training Data

**Important:** Update the paths below to point to your actual data directories

In [ ]:
# Colab paths - after extraction from Google Drive
TRAIN_DIR = '/content/CelebDF/train'
VAL_DIR = '/content/CelebDF/validation'
BATCH_SIZE = 32
IMG_SIZE = (256, 256)

## Create Data Generators and Load Data

In [ ]:
# Create data generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

# Load training data
print("Loading training data...")
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# Load validation data
print("Loading validation data...")
val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

print("✅ Data generators created successfully!")

## Setup Callbacks

In [ ]:
# Early stopping to prevent overfitting
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Save best model
checkpoint = keras.callbacks.ModelCheckpoint(
    '/content/drive/MyDrive/mesonet_best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

# Learning rate reduction
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

print("Callbacks configured")

## Train MesoNet Model

In [ ]:
EPOCHS = 30

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint, reduce_lr],
    verbose=1
)

print("\n✅ Training completed!")

## Plot Training History

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train')
axes[0, 0].plot(history.history['val_accuracy'], label='Validation')
axes[0, 0].set_title('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Loss
axes[0, 1].plot(history.history['loss'], label='Train')
axes[0, 1].plot(history.history['val_loss'], label='Validation')
axes[0, 1].set_title('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Precision
axes[1, 0].plot(history.history['precision'], label='Train')
axes[1, 0].plot(history.history['val_precision'], label='Validation')
axes[1, 0].set_title('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True)

# AUC
axes[1, 1].plot(history.history['auc'], label='Train')
axes[1, 1].plot(history.history['val_auc'], label='Validation')
axes[1, 1].set_title('AUC')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('AUC')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(r'c:\Users\Krish\project\New folder\Fake News and DeepFake detection system\mesonet_training_history.png', dpi=150)
plt.show()

## Evaluate Model on Validation Set

In [ ]:
# Evaluate on validation set
val_loss, val_acc, val_precision, val_recall, val_auc = model.evaluate(
    val_generator,
    verbose=1
)

print(f"\nValidation Metrics:")
print(f"  Accuracy:  {val_acc:.4f}")
print(f"  Precision: {val_precision:.4f}")
print(f"  Recall:    {val_recall:.4f}")
print(f"  AUC:       {val_auc:.4f}")
print(f"  Loss:      {val_loss:.4f}")

## Get Predictions and Classification Report

In [ ]:
# Get predictions
val_generator.reset()
predictions = model.predict(val_generator, verbose=1)
y_pred = (predictions > 0.5).astype(int).flatten()
y_true = val_generator.classes

# Classification report
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)

## Save Model for Backend Integration

In [ ]:
# Save model in multiple formats
model_path = r'c:\Users\Krish\project\New folder\Fake News and DeepFake detection system\mesonet_deepfake_detector.h5'
model.save(model_path)
print(f"✅ Model saved as: mesonet_deepfake_detector.h5")

# Also save as SavedModel format
saved_model_path = r'c:\Users\Krish\project\New folder\Fake News and DeepFake detection system\mesonet_saved_model'
model.save(saved_model_path)
print(f"✅ Model saved as: mesonet_saved_model")

# Save model info
model_info = {
    "model_name": "MesoNet Deepfake Detector",
    "input_shape": [256, 256, 3],
    "output_shape": 1,
    "classes": {"0": "Real", "1": "Fake"},
    "threshold": 0.5,
    "validation_accuracy": float(val_acc),
    "validation_auc": float(val_auc)
}

with open(r'c:\Users\Krish\project\New folder\Fake News and DeepFake detection system\mesonet_model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("✅ Model info saved")

## Test Inference on Sample Image

In [ ]:
# Load a sample image and test prediction
import random
from pathlib import Path

# Get a random test image
test_images = list(Path(VAL_DIR).rglob('*.jpg'))
test_image_path = random.choice(test_images)

# Load and preprocess
img = cv2.imread(str(test_image_path))
img_resized = cv2.resize(img, (256, 256))
img_normalized = img_resized / 255.0
img_batch = np.expand_dims(img_normalized, axis=0)

# Predict
prediction = model.predict(img_batch, verbose=0)[0][0]
confidence = prediction if prediction > 0.5 else 1 - prediction
label = "FAKE" if prediction > 0.5 else "REAL"

# Display
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title(f"Predicted: {label} (Confidence: {confidence:.2%})")
plt.axis('off')
plt.tight_layout()
plt.savefig(r'c:\Users\Krish\project\New folder\Fake News and DeepFake detection system\mesonet_sample_prediction.png', dpi=150)
plt.show()

print(f"\nTest Image: {test_image_path.name}")
print(f"Predicted: {label}")
print(f"Confidence: {confidence:.2%}")

## Backend Integration Code

Use this code in your FastAPI backend's `/analyze_image` endpoint

In [ ]:
# Backend integration example
backend_code = '''
# In your FastAPI backend (app.py)

from tensorflow import keras
import cv2
import numpy as np

# Load model once at startup
mesonet_model = keras.models.load_model('mesonet_deepfake_detector.h5')

@app.post("/analyze_image")
async def analyze_image(file: UploadFile = File(...)):
    """
    Analyze image for deepfake detection using MesoNet
    Returns: {"label": "FAKE" or "REAL", "confidence": 0.0-1.0}
    """
    # Read image
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    # Preprocess
    img_resized = cv2.resize(img, (256, 256))
    img_normalized = img_resized / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)
    
    # Predict
    prediction = mesonet_model.predict(img_batch, verbose=0)[0][0]
    confidence = float(prediction) if prediction > 0.5 else float(1 - prediction)
    label = "FAKE" if prediction > 0.5 else "REAL"
    
    return {
        "label": label,
        "confidence": confidence,
        "raw_prediction": float(prediction)
    }
'''

print(backend_code)

## Summary

✅ **Trained MesoNet model on:**
- 15,570 training images (7,785 real + 7,785 fake)
- 3,894 validation images (1,947 real + 1,947 fake)
- From 59 real FaceForensics++ deepfake videos

✅ **Saved files:**
- `mesonet_deepfake_detector.h5` - Trained model
- `mesonet_model_info.json` - Model metadata
- `mesonet_training_history.png` - Training graphs
- `mesonet_sample_prediction.png` - Example prediction

✅ **Next steps:**
1. Download the `.h5` model file
2. Add to your backend `/analyze_image` endpoint
3. Test with React frontend
4. Integrate with text analysis from Step 4

**This completes Step 5: Image Deepfake Detection! 🚀**